# Study 810 — Price Delay — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_weeks': 804, 'spread_bps': 2.45, 't_nw': 0.41, 't_1s': 0.43, 'long_bps': 36.73, 'short_bps': 34.28, 'welch_t': 0.2, 'gross_sharpe': 0.11, 'placebo_obs': 2.45, 'placebo_mean': -0.057, 'placebo_sd': 4.684, 'placebo_p': 0.301, 'placebo_draws': 1000, 'era_early_bps': 1.04, 'era_early_t': 0.17, 'era_early_n': 360, 'era_late_bps': 3.75, 'era_late_t': 0.39, 'era_late_n': 443, 'timer_1_gross': 2.45, 'timer_1_cost': 2.96, 'timer_1_net': -0.51, 'timer_1_t': -0.09, 'timer_5_gross': 2.45, 'timer_5_cost': 10.96, 'timer_5_net': -8.51, 'timer_5_t': -1.49, 'null_mean_t': 0.15, 'null_sd_t': 1.22, 'null_fire': 1, 'planted_t': 10.83, 'planted_welch': 1.79, 'fingerprint': '357fd262912f'}

## The headline — long-high-delay / short-low-delay spread

Weekly equal-weight top-30% minus bottom-30% price-delay spread (trailing-52-week delay, contemporaneous market + 4 weekly lags).

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/week  NW(6) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-delay {R['long_bps']:+.2f} vs low-delay {R['short_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : +2.45 bps/week  NW(6) t = +0.41  one-sample t = +0.43
books         : high-delay +36.73 vs low-delay +34.28 bps (Welch t = +0.20)
gross Sharpe  : 0.11 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

Keep the delay sort, but read each week's forward return from a column-permuted panel (signal → outcome link broken). If the edge were real the observed spread would sit far in the right tail; here it does not.

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> p = {R['placebo_p']:.5f}")

observed +2.45 bps vs placebo mean -0.057 (sd 4.684) -> p = 0.30100


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=360): +1.04 bps  NW t = +0.17
2018-2026 (n=443): +3.75 bps  NW t = +0.39


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per **week** on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/week (cost {c:.2f}/week, t={t:+.2f})")

 1 bp one-way: gross +2.45 -> net -0.51 bps/week (cost 2.96/week, t=-0.09)
5 bps one-way: gross +2.45 -> net -8.51 bps/week (cost 10.96/week, t=-1.49)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted premium.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from price_delay import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(knob=0.0, seed=811+s, n_assets=40, n_days=2000))['t_nw'] for s in range(8)])
print(f"null (knob=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(knob=0.0018, seed=810, n_assets=40, n_days=2000))
print(f"planted (knob=0.0018): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (knob=0), 8 seeds: NW t mean -0.14 (sd 0.95), |t|>=2 in 0/8


planted (knob=0.0018): NW t = +10.83, Welch t = +1.79


## Verdict

- **Signal — None.** The claimed Hou-Moskowitz delay premium does **not** replicate on 50 liquid US mega-caps: the long-high-delay / short-low-delay spread is **+2.45 bps/week** (NW *t* = **+0.41**) — the right sign but a coin-flip (placebo p ≈ 0.30), flat in both eras (*t* = +0.17 / +0.39). The 20-seed synthetic control recovers a *planted* premium cleanly (*t* = +10.83, fires on ~1/20 nulls, the expected 5% false-positive rate), so the null result is real, not machinery. The delay premium is a small / illiquid / neglected-stock effect; mega-caps are exactly where it should not appear. Survivorship biases the magnitude upward.
- **Tradability — Mirage.** Even the tiny positive gross edge dies on contact with costs: at 1 bp one-way the weekly friction (2.96 bps/week) already exceeds the 2.45 bps gross, net **-0.51 bps/week** (*t* = -0.09); at 5 bps **-8.51 bps/week**.